# Superstore PySpark Pipeline — v4

**Goal:** Rebuild the core Superstore data pipeline logic from v3 (Python/Pandas) using PySpark.
Same dataset, same business questions, same verified output — but now running on Spark's distributed engine.

**Topics covered:**
- SparkSession setup
- Reading CSV with schema handling
- select, filter, groupBy, agg
- withColumn, when(), col()
- Window functions (rank, dense_rank, running totals)
- Joins (inner, left, right, outer, broadcast)
- Writing output to Parquet and CSV

**Dataset:** Sample Superstore (9,994 records) — Telecom/Retail sales data  
**Stack:** PySpark 4.1.2, Python 3.13, Java 17

## 1. Environment Setup & SparkSession

In [ ]:
import os

# Force Spark to bind to localhost — avoids IP binding issues on laptops
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"
os.environ["PATH"] = "/opt/homebrew/opt/openjdk@17/bin:" + os.environ["PATH"]

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Superstore_v4")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print(f"Spark version: {spark.version}")
print("SparkSession ready.")

## 2. Load Data

Read the Superstore CSV. Key notes:
- `inferSchema=True` can mis-infer numeric columns as strings — we cast explicitly after loading
- Dataset has quoted fields with commas (e.g. product names) — use `quote` and `escape` options
- File uses latin1 encoding

In [ ]:
from pyspark.sql.functions import col

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", True)
    .csv("Sample - Superstore.csv")
)

# Cast numeric columns explicitly — inferSchema can get these wrong
df = df \
    .withColumn("Sales", col("Sales").cast("double")) \
    .withColumn("Profit", col("Profit").cast("double")) \
    .withColumn("Discount", col("Discount").cast("double")) \
    .withColumn("Quantity", col("Quantity").cast("integer"))

print(f"Total records: {df.count()}")
df.printSchema()

## 3. Basic Transformations — select, filter, groupBy, agg

Core DataFrame operations — equivalent to pandas `[[cols]]`, boolean indexing, and `groupby().agg()`.

In [ ]:
# select — pick specific columns
df.select("Category", "Sales", "Profit", "Discount").show(5)

In [ ]:
# filter — SQL WHERE equivalent
df.filter(col("Sales") > 500).select("Category", "Sales", "Profit").show(5)

# filter with multiple conditions
df.filter(
    (col("Category") == "Furniture") & (col("Profit") < 0)
).select("Category", "Sub-Category", "Sales", "Profit").show(5)

In [ ]:
import pyspark.sql.functions as F

# groupBy + agg — recreating category_summary() from v3 pandas pipeline
category_summary = df.groupBy("Category").agg(
    F.round(F.sum("Sales"), 2).alias("Total Sales"),
    F.round(F.sum("Profit"), 2).alias("Total Profit"),
    F.round(F.avg("Discount"), 4).alias("Avg Discount"),
    F.count("Order ID").alias("Total Orders")
).withColumn(
    "Profit_Perc",
    F.round((col("Total Profit") / col("Total Sales")) * 100, 2)
)

category_summary.show()

In [ ]:
# Region summary
region_summary = df.groupBy("Region").agg(
    F.round(F.sum("Sales"), 2).alias("Total Sales"),
    F.round(F.sum("Profit"), 2).alias("Total Profit"),
    F.count("Order ID").alias("Total Orders")
).orderBy(col("Total Sales").desc())

region_summary.show()

## 4. Enrichment — withColumn, when(), col()

Adding derived columns using Spark's native conditional functions.
Equivalent to pandas `apply()` with custom functions from v3.

In [ ]:
from pyspark.sql.functions import when

# Discount Risk — mirrors discount_risk() helper from v3
df = df.withColumn(
    "Discount_Risk",
    when(col("Discount") >= 0.30, "High Risk")
    .when(col("Discount") >= 0.15, "Medium Risk")
    .otherwise("Safe")
)

# Profit Category — mirrors profit_category() helper from v3
df = df.withColumn(
    "Profit_Category",
    when(col("Sales") == 0, "Unknown")
    .when((col("Profit") / col("Sales") * 100) > 20, "High Profit")
    .when((col("Profit") / col("Sales") * 100) >= 0, "Break Even")
    .otherwise("Loss")
)

df.select("Category", "Sales", "Profit", "Discount", "Discount_Risk", "Profit_Category").show(10)

## 5. Window Functions

Window functions operate over a partition of rows without collapsing them (unlike groupBy).
Equivalent to SQL `PARTITION BY ... ORDER BY`.

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, dense_rank, row_number

# Rank products by profit within each category
window_rank = Window.partitionBy("Category").orderBy(col("Profit").desc())

df_ranked = df.withColumn("profit_rank", rank().over(window_rank)) \
              .withColumn("dense_rank", dense_rank().over(window_rank))

# Top 3 products by profit per category
df_ranked.filter(col("profit_rank") <= 3) \
         .select("Category", "Product Name", "Profit", "profit_rank", "dense_rank") \
         .orderBy("Category", "profit_rank") \
         .show(10, truncate=False)

In [ ]:
# Running total of sales by region
window_running = (
    Window.partitionBy("Region")
    .orderBy("Order Date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df.withColumn("Running_Sales", F.sum("Sales").over(window_running)) \
  .select("Region", "Order Date", "Sales", "Running_Sales") \
  .show(10)

In [ ]:
# Average sales per category without collapsing rows (window vs groupBy)
window_avg = Window.partitionBy("Category")

df.withColumn("Avg_Category_Sales", F.avg("Sales").over(window_avg)) \
  .select("Category", "Sales", "Avg_Category_Sales") \
  .show(10)

## 6. Spark SQL

Register the DataFrame as a temporary view and query it with pure SQL.
Useful for complex analytics and for teams more comfortable with SQL than the DataFrame API.

In [ ]:
# Register as temp view
df.createOrReplaceTempView("superstore")

# Recreate category summary in pure SQL
spark.sql("""
    SELECT
        Category,
        ROUND(SUM(Sales), 2)          AS total_sales,
        ROUND(SUM(Profit), 2)         AS total_profit,
        COUNT(`Order ID`)             AS total_orders,
        ROUND(AVG(Discount), 4)       AS avg_discount,
        ROUND(SUM(Profit)/SUM(Sales)*100, 2) AS profit_margin_pct
    FROM superstore
    GROUP BY Category
    ORDER BY profit_margin_pct DESC
""").show()

In [ ]:
# Discount risk using CASE WHEN in SQL
spark.sql("""
    SELECT
        `Order ID`,
        Category,
        Sales,
        Discount,
        CASE
            WHEN Discount >= 0.30 THEN 'High Risk'
            WHEN Discount >= 0.15 THEN 'Medium Risk'
            ELSE 'Safe'
        END AS Discount_Risk
    FROM superstore
    LIMIT 10
""").show()

## 7. Joins

Joining the category summary against a business targets reference table.
This recreates the `margin_vs_expected()` logic from v3 — same output, verified against the original.

**Join types covered:** inner, left, right, full outer, broadcast

In [ ]:
from pyspark.sql import Row

# Business targets reference table (dimension table equivalent)
targets = spark.createDataFrame([
    Row(Category="Furniture",       expected_margin=15.0),
    Row(Category="Office Supplies", expected_margin=20.0),
    Row(Category="Technology",      expected_margin=25.0),
])

# Rebuild category summary with Profit_Perc for join
category_summary = df.groupBy("Category").agg(
    F.sum("Sales").alias("Total Sales"),
    F.sum("Profit").alias("Total Profit")
).withColumn(
    "Profit_Perc",
    (col("Total Profit") / col("Total Sales")) * 100
)

print("Category Summary:")
category_summary.show()
print("Targets:")
targets.show()

In [ ]:
# Inner Join — only matching rows from both sides
print("INNER JOIN:")
category_summary.join(targets, on="Category", how="inner").show()

# Left Join — all rows from left, matched from right
print("LEFT JOIN:")
category_summary.join(targets, on="Category", how="left").show()

# Right Join
print("RIGHT JOIN:")
category_summary.join(targets, on="Category", how="right").show()

# Full Outer Join
print("FULL OUTER JOIN:")
category_summary.join(targets, on="Category", how="outer").show()

In [ ]:
from pyspark.sql.functions import broadcast

# Broadcast Join — optimal when one table is small (dimension/lookup table)
# Spark sends the small table to every executor instead of shuffling both
result = (
    category_summary
    .join(broadcast(targets), on="Category", how="left")
    .withColumn("GAP", col("Profit_Perc") - col("expected_margin"))
    .withColumn("Status", when(col("GAP") > 0, "Passed").otherwise("Failed"))
)

result.select(
    "Category", "Total Sales", "Total Profit",
    "Profit_Perc", "expected_margin", "GAP", "Status"
).show()

# Note: Output matches v3 pandas pipeline exactly
# Furniture: -12.51 | Office Supplies: -2.96 | Technology: -7.60

## 8. Writing Output

Writing the final result to Parquet (standard for data lakes) and CSV (for compatibility).
- `coalesce(1)` merges partitions into a single output file — practical for local/small datasets
- Always specify `header=True` on both write AND read independently

In [ ]:
import os
os.makedirs("output", exist_ok=True)

# Write aggregated result as Parquet
result.write.mode("overwrite").parquet("output/margin_analysis_parquet")
print("Written: output/margin_analysis_parquet")

# Write aggregated result as CSV
result.coalesce(1) \
    .write.mode("overwrite") \
    .option("header", True) \
    .csv("output/margin_analysis_csv")
print("Written: output/margin_analysis_csv")

In [ ]:
# Verify Parquet read-back
print("Parquet verification:")
spark.read.parquet("output/margin_analysis_parquet").show()

# Verify CSV read-back — must specify header=True on read too
print("CSV verification:")
spark.read.option("header", True).csv("output/margin_analysis_csv").show()

## Summary

This notebook rebuilt the core Superstore pipeline logic from v3 (Python/Pandas) in PySpark.

**Key verified output — matches v3 exactly:**

| Category | Profit Margin % | Expected Target | GAP | Status |
|---|---|---|---|---|
| Furniture | 2.49% | 15% | -12.51 | Failed |
| Office Supplies | 17.04% | 20% | -2.96 | Failed |
| Technology | 17.40% | 25% | -7.60 | Failed |

**Next steps:**
- UDFs (User Defined Functions)
- Null handling
- Schema enforcement with StructType
- Repartition vs Coalesce
- Caching and persistence
- Delta Lake

**Repo:** https://github.com/surabhiks2k/superstore-data-platform